
**Title:** Regression Analysis for Flood Prediction Using LightGBM

**Abstract:**
This paper presents a regression analysis approach for predicting flood probabilities based on various environmental factors. The methodology employs LightGBM, a gradient boosting framework, to model the relationship between the input features and flood probabilities. The dataset used in this study includes environmental indicators such as Monsoon Intensity, Topography Drainage, River Management, and others. The results demonstrate the effectiveness of the proposed approach in accurately predicting flood probabilities, as evidenced by the achieved R-squared scores.

**Introduction:**
Flood prediction is a crucial task in disaster management and mitigation efforts. Accurate forecasting of flood probabilities can aid in proactive decision-making and resource allocation. Traditional regression analysis techniques have limitations in handling complex relationships between environmental factors and flood probabilities. In this study, we explore the use of LightGBM, a powerful gradient boosting framework, for regression analysis in flood prediction. LightGBM offers advantages such as high accuracy, scalability, and efficiency, making it well-suited for modeling complex datasets.

**Methodology:**
1. **Data Collection and Preprocessing:** The dataset comprises various environmental indicators, including Monsoon Intensity, Topography Drainage, River Management, and others. Data preprocessing involves handling missing values, standardizing features, and creating additional features based on the provided indicators.
   
2. **Model Development:** We employ LightGBM, a gradient boosting framework, for regression analysis. The model is trained on the training dataset using K-Fold cross-validation to ensure robustness and generalizability. Hyperparameter optimization is performed using Optuna to enhance the model's performance.

3. **Evaluation Metrics:** The performance of the regression model is evaluated using the R-squared metric, which measures the proportion of variance in the target variable that is predictable from the input features. A higher R-squared score indicates better predictive performance.

**Results:**
The experimental results demonstrate the effectiveness of the proposed approach in predicting flood probabilities. The LightGBM regression model achieves high R-squared scores on the validation dataset, indicating a strong correlation between the input features and the target variable. Furthermore, the model generalizes well to unseen data, as evidenced by consistent performance across multiple folds of cross-validation.

**Discussion:**
The use of LightGBM for regression analysis in flood prediction offers several advantages over traditional methods. Its ability to capture complex relationships and handle large-scale datasets makes it a valuable tool for disaster management authorities and policymakers. However, further research is needed to explore the impact of additional environmental factors and improve the model's interpretability.

**Conclusion:**
In conclusion, this study demonstrates the efficacy of LightGBM in regression analysis for flood prediction. By leveraging advanced machine learning techniques, accurate forecasts of flood probabilities can be obtained, aiding in proactive decision-making and disaster mitigation efforts. Future research directions may focus on incorporating real-time data streams and integrating ensemble learning methods to enhance predictive accuracy further.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Importing necessary libraries
import pandas as pd  # For data manipulation and analysis
import numpy as np  # For numerical computations
import matplotlib.pyplot as plt  # For data visualization
import seaborn as sns  # For enhanced data visualization

from lightgbm import LGBMRegressor  # LightGBM regressor model
from sklearn.model_selection import KFold  # For K-fold cross-validation
from sklearn.metrics import r2_score  # Evaluation metric: R-squared score
from sklearn.preprocessing import StandardScaler  # For feature scaling
import optuna  # For hyperparameter optimization
from optuna.samplers import TPESampler  # Tree-structured Parzen Estimator sampler
import warnings  # For managing warnings
import shap  # For SHAP (SHapley Additive exPlanations) values interpretation
warnings.filterwarnings("ignore")  # Suppressing warnings

In [ ]:
# Reading the training data from a CSV file
train = pd.read_csv('/kaggle/input/playground-series-s4e5/train.csv')

# Displaying the first few rows of the training data
train.head()

In [ ]:
# Checking for null values and data types in the training data
print("Null values in training data:\n", train.isnull().sum())
print("\nData types in training data:\n", train.dtypes)

In [ ]:
train.columns

In [ ]:
# Reading the test data from a CSV file
test = pd.read_csv('/kaggle/input/playground-series-s4e5/test.csv')

# Displaying the first few rows of the test data
test.head()

In [ ]:
# Checking for null values and data types in the test data
print("Null values in test data:\n", test.isnull().sum())
print("\nData types in test data:\n", test.dtypes)

In [ ]:
test.columns

In [ ]:
# Calculating the correlation matrix
correlation_matrix = train.corr()

# Plotting the correlation matrix
plt.figure(figsize=(20, 18))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.show()

1. **Documentation**: Comments provide explanations about the code's functionality, making it easier for others (and your future self) to understand what the code does.
2. **Clarity**: Comments clarify complex parts of the code, making it more readable and maintainable.
3. **Debugging**: Comments can help in debugging by providing insights into the intentions behind specific code sections.
4. **Collaboration**: When working in a team, comments facilitate collaboration by allowing team members to understand each other's code better.
5. **Learning**: Comments can serve as educational tools, helping others learn from your code and encouraging good coding practices.


In [ ]:
# Creating a figure for the histogram plot with specified size
plt.figure(figsize=(10, 4))

# Plotting the histogram of FloodProbability from the training data with specified bins
plt.hist(train.FloodProbability, bins=100)

# Adding a title to the plot
plt.title("Train data")

# Displaying the plot
plt.show()


1. **Figure Size**: The comment explains that `plt.figure(figsize=(10, 4))` is used to create a figure with a specific size, making it easier for someone reading the code to understand why this line is included.
  
2. **Histogram Plot**: The comment clarifies that `plt.hist(train.FloodProbability, bins=100)` is plotting a histogram of the FloodProbability feature from the training data, and it specifies the number of bins used in the histogram.

3. **Title**: The comment provides context for `plt.title("Train data")`, indicating that it sets the title of the plot to "Train data", which helps the reader understand what the plot represents.

4. **Display**: Finally, the comment explains `plt.show()`, indicating that it displays the plot to the user.


In [ ]:
# List of columns that are not features
NON_FEATURES = ['id', 'FloodProbability', 'fold']

# List of base features used for creating additional features
BASE_FEATURES = ['MonsoonIntensity', 'TopographyDrainage', 'RiverManagement',
       'Deforestation', 'Urbanization', 'ClimateChange', 'DamsQuality',
       'Siltation', 'AgriculturalPractices', 'Encroachments',
       'IneffectiveDisasterPreparedness', 'DrainageSystems',
       'CoastalVulnerability', 'Landslides', 'Watersheds',
       'DeterioratingInfrastructure', 'PopulationScore', 'WetlandLoss',
       'InadequatePlanning', 'PoliticalFactors']

# Function to add additional features based on base features
def add_features(df):
    df['total'] = df[BASE_FEATURES].sum(axis=1)
    df['mean'] = df[BASE_FEATURES].mean(axis=1)
    df['std'] = df[BASE_FEATURES].std(axis=1)
    df['max'] = df[BASE_FEATURES].max(axis=1)
    df['min'] = df[BASE_FEATURES].min(axis=1)
    df['median'] = df[BASE_FEATURES].median(axis=1)
    df['ptp'] = df[BASE_FEATURES].values.ptp(axis=1)
    df['q25'] = df[BASE_FEATURES].quantile(0.25, axis=1)
    df['q75'] = df[BASE_FEATURES].quantile(0.75, axis=1)
    return df

# Adding additional features to the training data
train = add_features(train)

# List of all features including the additional ones
FEATURES = [col for col in train.columns if col not in NON_FEATURES]

# Reordering the columns of the training data
train = train[['id'] + FEATURES + ['FloodProbability']]

# Adding additional features to the test data
test = add_features(test)

# Extracting the features for testing
X_test = test[FEATURES]

# Extracting features and target variable for training
X_train = train.drop(['id', 'FloodProbability'], axis=1)
y_train = train['FloodProbability']

# Standardizing features using StandardScaler
s = StandardScaler()
X_train = s.fit_transform(X_train)
X_test = s.transform(X_test)

 explain each part:

1. **Non-Feature Columns** (`NON_FEATURES`):
   - This list contains the column names that are not considered as features for modeling. These typically include identifiers (`id`), target variable (`FloodProbability`), and any fold information used for cross-validation (`fold`).

2. **Base Features** (`BASE_FEATURES`):
   - This list contains the names of columns that serve as the base features for creating additional features. These features are likely relevant predictors for flood prediction.

3. **Function to Add Additional Features** (`add_features`):
   - This function takes a DataFrame (`df`) as input and adds additional features based on the base features defined earlier.
   - The additional features include: total sum, mean, standard deviation, maximum, minimum, median, peak-to-peak (range), and quartiles (q25 and q75) calculated across the base features.
   - These additional features aim to capture different statistical properties and characteristics of the base features, potentially enhancing the predictive power of the model.

4. **Adding Additional Features to Training and Test Data**:
   - The `add_features` function is applied to both the training and test datasets, enriching them with the calculated additional features.

5. **Extracting Features and Target Variable**:
   - The feature matrix `X_train` is extracted from the training data by dropping non-feature columns (`id` and `FloodProbability`).
   - The target variable `y_train` is extracted from the training data.
   - Similarly, features for testing (`X_test`) are extracted from the test data.

6. **Standardizing Features**:
   - The features are standardized using `StandardScaler`. This step ensures that all features have a mean of 0 and a standard deviation of 1, which can improve the performance of some machine learning algorithms.


In [ ]:
# Creating a KFold cross-validation object with 5 folds, shuffling the data, and setting a random state
cv = KFold(5, shuffle=True, random_state=0)

# Generating indices for training and validation sets for each fold
cv_splits = cv.split(X_train, y_train)

# Initializing an empty list to store R-squared scores for each fold
scores = list()

# Initializing the LightGBM regression model
model = LGBMRegressor(objective='regression', random_state=0, device='cpu', verbosity=-1,)

# Looping over each fold of the cross-validation
for train_idx, val_idx in cv_splits:
    # Splitting the data into training and validation sets for the current fold
    X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
    y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]
    
    # Fitting the model on the training data for the current fold
    model.fit(X_train_fold, y_train_fold)
    
    # Making predictions on the validation data for the current fold
    y_pred = model.predict(X_val_fold)
    
    # Calculating the R-squared score for the current fold and appending it to the scores list
    r2 = r2_score(y_val_fold, y_pred)
    scores.append(r2)

# Calculating and printing the mean R-squared score across all folds
print(f'Mean R2 score: {np.mean(scores):.5f}')

the code snippet:

1. **KFold Cross-Validation Object**: 
   - `cv = KFold(5, shuffle=True, random_state=0)`: This creates a KFold cross-validation object with 5 folds (`n_splits=5`). The `shuffle=True` parameter shuffles the data before splitting, and `random_state=0` ensures reproducibility by fixing the random seed.

2. **Generating Cross-Validation Splits**:
   - `cv_splits = cv.split(X_train, y_train)`: This generates indices for splitting the training data (`X_train`) and target variable (`y_train`) into training and validation sets for each fold of cross-validation.

3. **Initialization**:
   - `scores = list()`: This initializes an empty list to store the R-squared scores for each fold.

4. **Model Initialization**:
   - `model = LGBMRegressor(objective='regression', random_state=0, device='gpu', verbosity=-1,)`: This initializes a LightGBM regression model. Parameters like `objective`, `random_state`, `device`, and `verbosity` are specified. The model will perform regression (`objective='regression'`), use GPU for computation (`device='gpu'`), and suppress verbosity (`verbosity=-1`).

5. **Cross-Validation Loop**:
   - `for train_idx, val_idx in cv_splits:`: This iterates over each fold of the cross-validation. `train_idx` and `val_idx` represent the indices of the training and validation sets for the current fold, respectively.

6. **Data Splitting**:
   - `X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]`: This splits the features into training and validation sets for the current fold.
   - `y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]`: This splits the target variable into training and validation sets for the current fold.

7. **Model Training and Evaluation**:
   - `model.fit(X_train_fold, y_train_fold)`: This fits the model on the training data for the current fold.
   - `y_pred = model.predict(X_val_fold)`: This makes predictions on the validation data for the current fold.
   - `r2 = r2_score(y_val_fold, y_pred)`: This calculates the R-squared score for the current fold.
   - `scores.append(r2)`: This appends the R-squared score to the `scores` list.

8. **Mean R-squared Score**:
   - `print(f'Mean R2 score: {np.mean(scores):.5f}')`: This calculates and prints the mean R-squared score across all folds, providing an overall assessment of the model's performance.


In [ ]:
# Set optimization flag (True for hyperparameter optimization, False for using predefined parameters)
optimize = False

# Objective function for hyperparameter optimization
def objective(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 1.0, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 300, 1200),
        'subsample_for_bin': trial.suggest_int('subsample_for_bin', 20000, 300000),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 500),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-9, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-9, 10.0, log=True),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'subsample': trial.suggest_float('subsample', 0.25, 1.0),
        'max_depth': trial.suggest_int('max_depth', 1, 15)
    }
    
    cv = KFold(5, shuffle=True, random_state=0)
    cv_splits = cv.split(X_train, y_train)
    scores = list()

    for train_idx, val_idx in cv_splits:
        X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
        y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]
        model = LGBMRegressor(**params, objective='regression', random_state=0, device='gpu', verbosity = -1,)
        model.fit(X_train_fold, y_train_fold)
        
        y_pred = model.predict(X_val_fold)
        r2 = r2_score(y_val_fold, y_pred)
        scores.append(r2)
        
    return np.mean(scores)

# SQLite database for storing optimization results
sqlite_db = "sqlite:///lgbm1.db"

# Study name for the optimization study
study_name = "lgbm"

# Hyperparameter optimization process
if optimize:
    # Create an Optuna study for hyperparameter optimization
    study = optuna.create_study(storage=sqlite_db, study_name=study_name, 
                                sampler=TPESampler(n_startup_trials=75, multivariate=True, seed=0),
                                direction="maximize", load_if_exists=True)

    # Perform hyperparameter optimization
    study.optimize(objective, n_trials=200)
    
    # Print the best optimized accuracy and hyperparameters
    print(f"best optimized accuracy: {study.best_value:0.5f}")
    print(f"best hyperparameters: {study.best_params}")

    # Retrieve the best hyperparameters found during optimization
    lgbm_params = study.best_params
else: 
    # Use predefined parameters if optimization flag is set to False
    lgbm_params = {
        'num_leaves': 183, 
        'learning_rate': 0.01183688880802108, 
        'n_estimators': 577, 
        'subsample_for_bin': 165697, 
        'min_child_samples': 114, 
        'reg_alpha': 2.075080888948164e-06, 
        'reg_lambda': 3.838938366471552e-07, 
        'colsample_bytree': 0.9634044234652241, 
        'subsample': 0.9592138618622019, 
        'max_depth': 9
    }


1. **Optimization Flag** (`optimize = False`):
   - This variable determines whether hyperparameter optimization should be performed (`True`) or if predefined parameters should be used (`False`).

2. **Objective Function for Hyperparameter Optimization** (`objective`):
   - This function defines the objective to optimize during hyperparameter tuning using Optuna.
   - It takes a trial object as input, which represents a single execution of the optimization process.
   - Inside the function, a set of hyperparameters is defined based on suggested ranges provided by the trial.
   - The objective is to maximize the mean R-squared score obtained through cross-validation using the specified hyperparameters.

3. **SQLite Database for Storing Optimization Results** (`sqlite_db = "sqlite:///lgbm1.db"`):
   - This variable specifies the SQLite database file path where optimization results will be stored.

4. **Study Name for the Optimization Study** (`study_name = "lgbm"`):
   - This variable defines the name of the optimization study conducted using Optuna.

5. **Hyperparameter Optimization Process**:
   - If `optimize` is set to `True`, hyperparameter optimization using Optuna will be performed. Otherwise, predefined parameters will be used.
   - Inside the optimization block:
     - A study object is created to manage the optimization process.
     - The study's sampler, direction (maximize/minimize), and storage settings are specified.
     - The `objective` function is optimized for a specified number of trials (`n_trials`).
     - The best optimized accuracy and corresponding hyperparameters are printed.

6. **Predefined Parameters**:
   - If `optimize` is set to `False`, predefined hyperparameters are provided.
   - These parameters are used when hyperparameter optimization is disabled.



In [ ]:
# Creating a KFold cross-validation object with 5 folds, shuffling the data, and setting a random state
cv = KFold(5, shuffle=True, random_state=0)

# Generating indices for training and validation sets for each fold
cv_splits = cv.split(X_train, y_train)

# Initializing empty lists to store R-squared scores for each fold and predictions on test data
scores = list()
test_preds = list()

# Initializing the LightGBM regression model with the optimized parameters
model = LGBMRegressor(**lgbm_params, objective='regression', random_state=0, verbosity=-1)

# Looping over each fold of the cross-validation
for train_idx, val_idx in cv_splits:
    # Splitting the data into training and validation sets for the current fold
    X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
    y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]
    
    # Fitting the model on the training data for the current fold
    model.fit(X_train_fold, y_train_fold)
    
    # Making predictions on the validation data for the current fold
    y_val_prob = model.predict(X_val_fold)
    
    # Calculating the R-squared score for the current fold and appending it to the scores list
    r2 = r2_score(y_val_fold, y_val_prob)
    scores.append(r2)
    
    # Making predictions on the test data for the current fold and storing the predictions
    y_pred = model.predict(X_test)
    test_preds.append(y_pred)

# Calculating and printing the mean R-squared score across all folds
print(f'Mean R2 score: {np.mean(scores):.5f}')

Here's an explanation of the code:

1. **KFold Cross-Validation Setup**:
   - A `KFold` cross-validation object is created with 5 folds (`n_splits=5`), and the data is shuffled (`shuffle=True`) with a fixed random state (`random_state=0`).
   - The `split` method of the cross-validation object is then used to generate indices for splitting the training data (`X_train`) and target variable (`y_train`) into training and validation sets for each fold.

2. **Initialization**:
   - Empty lists `scores` and `test_preds` are initialized to store R-squared scores for each fold and predictions on the test data, respectively.
   - The LightGBM regression model (`LGBMRegressor`) is initialized with the optimized hyperparameters (`lgbm_params`) obtained from hyperparameter optimization or predefined values.
   
3. **Cross-Validation Loop**:
   - The loop iterates over each fold of the cross-validation:
     - It splits the training data into training and validation sets for the current fold (`X_train_fold`, `X_val_fold`, `y_train_fold`, `y_val_fold`).
     - The model is trained on the training data for the current fold using the `fit` method.
     - Predictions are made on the validation data (`X_val_fold`) using the `predict` method, and the R-squared score is calculated between the true validation labels (`y_val_fold`) and predicted values (`y_val_prob`).
     - The R-squared score for the current fold is appended to the `scores` list.
     - Predictions are made on the test data (`X_test`) for the current fold using the trained model, and the predictions are stored in the `test_preds` list.

4. **Results and Evaluation**:
   - After all folds are processed, the mean R-squared score across all folds is calculated using `np.mean(scores)` and printed.
   


In [ ]:
# Reading the sample submission file containing the template for submission
sample_submission = pd.read_csv('/kaggle/input/playground-series-s4e5/sample_submission.csv')

# Calculating the mean of predictions across all folds for each sample in the test data
sample_submission['FloodProbability'] = np.mean(test_preds, axis=0)

# Saving the submission dataframe to a CSV file without including the index
sample_submission.to_csv('submission.csv', index=False)

# Displaying the first few rows of the submission dataframe
sample_submission.head()

Here's an explanation of each part of the code:

1. **Reading Sample Submission File**:
   - The code reads the sample submission file (`sample_submission.csv`) using `pd.read_csv()`. This file likely contains the structure of the submission format required for the competition, with an 'id' column and a 'FloodProbability' column.

2. **Calculating Mean Predictions**:
   - After generating predictions on the test data for each fold (`test_preds`), the code calculates the mean of these predictions across all folds for each sample in the test data. This is achieved using `np.mean(test_preds, axis=0)`.
   - The resulting array of mean predictions is assigned to the 'FloodProbability' column of the `sample_submission` dataframe.

3. **Saving Submission File**:
   - The updated `sample_submission` dataframe with the mean predictions is saved to a CSV file named `submission.csv` using `to_csv()` method. The parameter `index=False` specifies that the index column should not be included in the CSV file.

4. **Displaying Submission DataFrame**:
   - Finally, the first few rows of the updated `sample_submission` dataframe are displayed using the `head()` method to verify the correctness of the submission format and the predictions.




1. **Data Loading and Inspection**:
   - Load the training and testing datasets using `pandas.read_csv()`.
   - Inspect the structure of the datasets, check for missing values, and understand the data types.

   Packages needed: pandas, numpy

2. **Data Visualization**:
   - Visualize the correlation matrix of the features in the training data using `seaborn.heatmap()`.
   - Plot histograms to understand the distribution of the target variable and other relevant features.

   Packages needed: matplotlib, seaborn

3. **Feature Engineering**:
   - Define non-feature columns and base feature columns.
   - Create additional features based on the base features, such as sum, mean, standard deviation, etc.

   Packages needed: None

4. **Data Preprocessing**:
   - Apply the created additional features to both the training and testing datasets.
   - Separate the features from the target variable.
   - Standardize the features using `StandardScaler` from `sklearn.preprocessing`.

   Packages needed: scikit-learn

5. **Model Training and Evaluation**:
   - Perform K-Fold cross-validation using `KFold` from `sklearn.model_selection`.
   - Initialize the LightGBM regressor model with optimized parameters.
   - Train the model on each fold of the cross-validation.
   - Evaluate the model performance using R-squared metric on the validation set.

   Packages needed: lightgbm, sklearn

6. **Hyperparameter Optimization (Optional)**:
   - If desired, perform hyperparameter optimization using `optuna`.
   - Define the objective function for optimization.
   - Optimize hyperparameters using Optuna's `TPESampler`.
   - Alternatively, you can directly specify hyperparameters based on prior optimization results.

   Packages needed: optuna

7. **Generating Predictions**:
   - Generate predictions on the test data for each fold of the cross-validation.
   - Average the predictions across all folds to obtain the final predictions.

   Packages needed: None

8. **Submission Preparation**:
   - Read the sample submission file.
   - Replace the target variable with the final predictions.
   - Save the submission file as a CSV without the index column.

   Packages needed: pandas

